# 🌽☕ AgriChain: High-Accuracy Disease Diagnosis Model
This notebook trains an advanced **MobileNetV2** model for disease diagnosis using **MindSpore 1.7+**.

### 📋 Instructions:
1. Ensure your dataset is mounted at `/data/agrichain/dataset.zip` using OBS.
2. Run all cells.
3. Download the resulting `.ms` files for your Flutter app.

## 🛠 Setup Environment

In [ ]:
import os
import zipfile
import shutil
import random
import numpy as np
from glob import glob
from PIL import Image
import mindspore as ms
import mindspore.dataset as ds
import mindspore.dataset.vision.c_transforms as vision
import mindspore.dataset.transforms.c_transforms as transforms
from mindspore import nn, Model, context
from mindspore.train.callback import LossMonitor

context.set_context(mode=context.GRAPH_MODE, device_target="GPU")
print(f"MindSpore Version: {ms.__version__} imported successfully!")

## 📦 Step 1: Prepare Dataset (Unzip, Clean, & Split)
This cell unzips your data, filters out **corrupted images** that break training, and splits the rest into Train/Val folders.

In [ ]:
def prepare_data(zip_path, extract_path='./data_raw', split_path='./datasets'):
    if os.path.exists(split_path): shutil.rmtree(split_path)
    
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("✅ Unzipped dataset successfully")
    else:
        print(f"❌ {zip_path} not found!")
        return
    
    root_search = os.path.join(extract_path, 'dataset') if os.path.exists(os.path.join(extract_path, 'dataset')) else extract_path
    
    for crop in ['maize_dataset', 'coffee_dataset']:
        crop_raw = os.path.join(root_search, crop)
        if not os.path.exists(crop_raw): continue
        
        classes = [d for d in os.listdir(crop_raw) if os.path.isdir(os.path.join(crop_raw, d))]
        print(f"\n⏳ Processing {crop}...")
        
        for cls in classes:
            raw_images = glob(os.path.join(crop_raw, cls, '*'))
            valid_images = []
            
            for img_path in raw_images:
                try:
                    with Image.open(img_path) as img:
                        img.verify()
                    valid_images.append(img_path)
                except Exception as e:
                    pass
            
            print(f"  - {cls}: {len(valid_images)} valid images")
            random.shuffle(valid_images)
            
            split_idx = int(len(valid_images) * 0.8)
            train_images = valid_images[:split_idx]
            val_images = valid_images[split_idx:]
            
            os.makedirs(os.path.join(split_path, crop, 'train', cls), exist_ok=True)
            os.makedirs(os.path.join(split_path, crop, 'val', cls), exist_ok=True)
            for img in train_images: shutil.copy(img, os.path.join(split_path, crop, 'train', cls))
            for img in val_images: shutil.copy(img, os.path.join(split_path, crop, 'val', cls))

dataset_path = '/data/agrichain/dataset.zip'

if os.path.exists(dataset_path):
    print(f"✅ Found dataset at: {dataset_path}")
    prepare_data(dataset_path)
else:
    print(f"❌ Dataset not found at {dataset_path}")
    print("\nLet's see what's in /data/agrichain/:")
    if os.path.exists('/data/agrichain/'):
        for file in os.listdir('/data/agrichain/'):
            print(f"  - {file}")

## 📊 Data Pipeline Factory

In [ ]:
import mindspore.common.dtype as mstype

def create_dataset(data_path, batch_size=32, training=True):
    dataset = ds.ImageFolderDataset(data_path, decode=True, shuffle=training)
    
    image_size = (224, 224)
    mean = [0.485 * 255, 0.456 * 255, 0.406 * 255]
    std = [0.229 * 255, 0.224 * 255, 0.225 * 255]
    
    if training:
        trans = [
            vision.Resize((256, 256)), 
            vision.RandomCrop(image_size),
            vision.RandomHorizontalFlip(prob=0.5),
            vision.RandomVerticalFlip(prob=0.2),
            vision.RandomColorAdjust(brightness=0.4, contrast=0.4, saturation=0.4),
            vision.Normalize(mean=mean, std=std),
            vision.HWC2CHW()
        ]
    else:
        trans = [
            vision.Resize((256, 256)),
            vision.CenterCrop(image_size),
            vision.Normalize(mean=mean, std=std),
            vision.HWC2CHW()
        ]

    dataset = dataset.map(operations=trans, input_columns="image", num_parallel_workers=4)
    dataset = dataset.map(operations=transforms.TypeCast(mstype.int32), input_columns="label", num_parallel_workers=4)
    dataset = dataset.batch(batch_size, drop_remainder=True)
    return dataset

## 🏗 High Accuracy MobileNetV2 Architecture

In [ ]:
from mindspore.common.initializer import Normal
from mindvision.classification.models import mobilenet_v2

def build_model(num_classes):
    print("Loading High-Accuracy MobileNetV2 architecture...")
    # Note: We use the mindvision API available in modern MindSpore versions
    # We instantiate it directly for the specific number of disease classes
    network = mobilenet_v2(num_classes=num_classes, pretrained=False)
    return network

## 🚀 Training & Export

In [ ]:
from mindspore import export

def run_training(name, num_classes, base_path, epochs=15):
    train_path = os.path.join(base_path, 'train')
    val_path = os.path.join(base_path, 'val')
    
    if not os.path.exists(train_path): return
    
    print(f"\n--- Configuring {name.upper()} Model ---")
    train_ds = create_dataset(train_path, batch_size=32, training=True)
    val_ds = create_dataset(val_path, batch_size=32, training=False)
    
    if train_ds.get_dataset_size() == 0:
        print(f"❌ ERROR: Train dataset for {name} is empty!")
        return

    net = build_model(num_classes)
    loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
    
    # Lower learning rate since MobileNetV2 is deep and needs careful tuning
    opt = nn.Adam(net.trainable_params(), learning_rate=0.0005)
    
    model = Model(net, loss_fn=loss, optimizer=opt, metrics={'accuracy': nn.Accuracy()})
    
    print(f"🚀 Training MobileNetV2 on {name} for {epochs} epochs...")
    model.train(epochs, train_ds, callbacks=[LossMonitor(per_print_times=train_ds.get_dataset_size())], dataset_sink_mode=False)
    
    print(f"\n📊 Evaluating {name} Validation Accuracy...")
    acc = model.eval(val_ds, dataset_sink_mode=False)
    print(f"✅ {name.upper()} FINAL ACCURACY: {acc['accuracy'] * 100:.2f}%")
    
    # Export to MindIR for Mobile deployment
    input_tensor = ms.Tensor(np.ones([1, 3, 224, 224]), ms.float32)
    export(net, input_tensor, file_name=f"{name}_disease", file_format="MINDIR")
    print(f"📦 Exported {name}_disease.mindir")

# 🟢 Start Process
run_training("maize", 4, './datasets/maize_dataset')
run_training("coffee", 3, './datasets/coffee_dataset')

## 📱 Convert to MindSpore Lite (.ms)
This final cell acts like `convert.py`. It takes the exported `.mindir` files and compiles them into highly optimized `.ms` format for the Flutter mobile application.

In [ ]:
import subprocess

def convert_to_lite(mindir_model, output_name):
    print(f"\n🔄 Submitting {mindir_model} to MindSpore Lite Converter...")
    
    # If it's going to an Android/Flutter phone, we want exactly this shape.
    # Note: converter_lite automatically append ".ms" to the outputFile name.
    cmd = (
        f"converter_lite "
        f"--fmk=MINDIR "
        f"--modelFile={mindir_model} "
        f"--outputFile={output_name} "
        f"--inputShape=\"1,3,224,224\""
    )
    
    # Run the bash command through python
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ SUCCESS: Created {output_name}.ms size: {os.path.getsize(f'{output_name}.ms') / (1024*1024):.2f} MB")
    else:
        print(f"❌ CONVERSION FAILED!")
        print(result.stderr)
        print(result.stdout)

print("========== STARTING MOBILE OPTIMIZATION ==========")

if os.path.exists('maize_disease.mindir'):
    convert_to_lite("maize_disease.mindir", "maize_disease_fp32")
else:
    print("⚠️ maize_disease.mindir not found. Did the training finish?")

if os.path.exists('coffee_disease.mindir'):
    convert_to_lite("coffee_disease.mindir", "coffee_disease_fp32")
else:
    print("⚠️ coffee_disease.mindir not found. Did the training finish?")

print("\n🏁 All done! You can now download the .ms files from the left panel.")